#Positional Encoding

In [1]:
import math
import torch
import torch.nn as nn

let d_model=8,batch=2,seq_len=5

In [6]:
class PositionalEncoding(nn.Module):
  def __init__(self,d_model,max_len=5000):
    super().__init__()

    position=torch.arange(
        max_len
    ).unsqueeze(1)

    div_term=torch.exp(
        torch.arange(0,d_model,2)*
        (-math.log(10000.0)/d_model)
    )

    pe=torch.zeros(max_len,d_model); #(max_len,d_model)

    #[All row and 0,2,4--- col ]
    pe[:,0::2]=torch.sin(
        position*div_term
    )
    #[All row and 1,3,5--]
    pe[:,1::2]=torch.cos(
        position*div_term
    )

    pe=pe.unsqueeze(0) #(batch,max_len,d_model)

    self.register_buffer(
        "pe",pe
    )

  def forward(self,x):
    x=x+self.pe[:,x.size(1)]
    return x


In [7]:
batch=2
seq_len=5
d_model=8
# input(2,5,8)
x=torch.randn(
    batch,seq_len,d_model
)
print(x)
print("="*50)
print(x.shape)

tensor([[[-1.0253,  1.0664, -1.5736,  0.8135,  0.0411,  0.8893, -0.9806,
          -0.0740],
         [-0.6485,  0.3173, -0.1877, -1.9361, -1.3676, -1.4620, -0.4859,
          -0.9897],
         [-1.4474,  2.4608,  1.4908,  0.5051, -0.0660,  0.4479,  2.0897,
          -0.4156],
         [ 0.3990,  0.6493,  0.4288,  0.3519,  1.3287,  0.2550, -1.6997,
          -0.6593],
         [ 0.3457,  1.4849, -0.4701, -0.3087, -0.3070, -0.9777, -0.1363,
           0.0230]],

        [[-0.3585, -0.1944, -0.3948,  0.4627, -0.0486, -1.1483,  0.9029,
           0.4547],
         [-0.9600, -0.1538,  1.5411, -0.7026, -0.5249, -0.5818,  0.4227,
           1.3653],
         [-0.8506,  0.1340,  0.0447,  0.3199, -1.1514, -0.8914, -0.4173,
          -0.6272],
         [ 0.1880,  0.1794,  0.6837, -1.1426,  3.0782, -0.2760,  1.3786,
          -0.5116],
         [ 0.6171, -0.4938, -0.6261, -1.2305, -0.2421, -0.5984,  0.6093,
          -1.0186]]])
torch.Size([2, 5, 8])


In [8]:
pe=PositionalEncoding(d_model)
output=pe(x)
print(output)
print("="*50)
print(output.shape)

tensor([[[-1.9842e+00,  1.3500e+00, -1.0942e+00,  1.6911e+00,  9.1123e-02,
           1.8881e+00, -9.7562e-01,  9.2597e-01],
         [-1.6075e+00,  6.0093e-01,  2.9168e-01, -1.0586e+00, -1.3177e+00,
          -4.6325e-01, -4.8092e-01,  1.0294e-02],
         [-2.4063e+00,  2.7444e+00,  1.9703e+00,  1.3826e+00, -1.6042e-02,
           1.4466e+00,  2.0947e+00,  5.8440e-01],
         [-5.5988e-01,  9.3300e-01,  9.0824e-01,  1.2295e+00,  1.3787e+00,
           1.2538e+00, -1.6947e+00,  3.4072e-01],
         [-6.1323e-01,  1.7686e+00,  9.3108e-03,  5.6891e-01, -2.5707e-01,
           2.1086e-02, -1.3129e-01,  1.0230e+00]],

        [[-1.3174e+00,  8.9300e-02,  8.4653e-02,  1.3403e+00,  1.3982e-03,
          -1.4955e-01,  9.0787e-01,  1.4546e+00],
         [-1.9189e+00,  1.2987e-01,  2.0205e+00,  1.7496e-01, -4.7493e-01,
           4.1695e-01,  4.2767e-01,  2.3652e+00],
         [-1.8095e+00,  4.1769e-01,  5.2411e-01,  1.1974e+00, -1.1014e+00,
           1.0731e-01, -4.1233e-01,  3.7279e-01]

## Dimesion Flow



```
              Embedding
              (2,5,8)
                    │
                    ▼
              Position Matrix
              (1,5000,8)
                    │
              Slice
                    ▼
              (1,5,8)
                    │
              Broadcast Add
                    ▼
              Final Input
              (2,5,8)
```






**register_buffer() কেন ব্যবহার করা হয়?**

উত্তর: কারণ Positional Encoding একটি fixed tensor। এটি trainable parameter নয়, কিন্তু model save/load এবং CPU↔GPU transfer-এর সময় model-এর সাথেই থাকতে হবে।

**unsqueeze(0) কেন করা হয়?**

উত্তর: Batch dimension যোগ করার জন্য, যাতে (batch_size, seq_len, d_model) embedding-এর সাথে broadcasting করে যোগ করা যায়।

**pe[:, :x.size(1)] কেন?**

উত্তর: কারণ সব sentence-এর length সমান নয়। max_len=5000 পর্যন্ত encoding তৈরি করা থাকে, কিন্তু বর্তমান batch-এ যতগুলো token আছে (x.size(1)), ততগুলো position-ই নেওয়া হয়






